In [16]:
import pandas as pd
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from docx import Document
from datetime import datetime

In [2]:
import os
import sys
sys.path.append('..')
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import dataframe_image as dfi
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine, text
from docxtpl import DocxTemplate, InlineImage
from docx.shared import Inches
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y BASE DE DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"

from config import POSTGRES_UTEA
POSTGRES_UTEA['DATABASE'] = 'utea_precision'

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# Carga de la matriz base CSV para los Combo Box
df = pd.read_csv(os.path.join(ruta_base, "DATOS_LOTES.csv"), encoding="utf-8")
df = df.rename(columns={'TCH Estiamdo': 'TCH Estimado'})
plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_F1.docx") 

# Paletas de colores sincronizadas para mapas y tablas
colores_velocidad = {'<4 km/h': '#e74c3c', '4-6.5 km/h': '#2ecc71', '>6.5 km/h': '#3498db'}
colores_elevacion = {'Bajo': '#e74c3c', 'Medio': '#2ecc71', 'Alto': '#3498db'}
colores_tch = {
    '< 40 Tn/Ha': '#e74c3c',       # Rojo
    '40 a 60 Tn/Ha': '#f1c40f',    # Amarillo
    '60 a 80 Tn/Ha': '#2ecc71',    # Verde
    '> 80 Tn/Ha': '#3498db'        # Azul
}

# --- 2. FUNCIONES DE PROCESAMIENTO GEOGRÁFICO Y GRÁFICOS ---

def obtener_recorrido_postgres(propiedad, lote):
    """Descarga los datos geográficos desde PostGIS incluyendo vryldrcane."""
    engine = obtener_engine()
    # Agregamos vryldrcane a la consulta SQL
    query = """
        SELECT id, vehiclspeed, elevation, vryldrcane, distance, swathwidth, distancia_real, geom 
        FROM datos_cosecha.recorrido_cosechadora 
        WHERE propiedad = %(prop)s AND lote = %(lote)s
    """
    try:
        gdf = gpd.read_postgis(query, engine, geom_col='geom', params={"prop": propiedad, "lote": lote})
        return gdf
    except Exception as e:
        print(f"❌ Error al extraer telemetría de Postgres: {e}")
        return gpd.GeoDataFrame()

def procesar_y_graficar(gdf, tipo, dict_colores, path_img, path_tabla):
    """Categoriza variables, genera planos (filtrados < 3) y cuadros estadísticos coordinados."""
    gdf = gdf.copy()
    
    # --- Categorización Dinámica General (Conservando tu lógica exacta para Elevación) ---
    if tipo == 'velocidad':
        gdf['categoria'] = pd.cut(gdf['vehiclspeed'], bins=[0, 4, 7, 100], labels=['<4 km/h', '4-6.5 km/h', '>6.5 km/h'])
        col_name = 'Velocidad'
        
    elif tipo == 'elevacion':
        # 1. Filtramos los registros mayores a 0
        gdf_elevacion_valida = gdf[gdf['elevation'] > 0]
        
        if not gdf_elevacion_valida.empty:
            # CLAVE: Calculamos los cortes dinámicos con pd.cut usando SOLO los datos > 0
            # include_lowest=True asegura que el decimal más bajo sea tomado en cuenta
            cortes = pd.cut(gdf_elevacion_valida['elevation'], bins=3, labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)
            
            # Asignamos esos cortes al gdf original. Las filas que tenían '0' se volverán automáticamente NaN
            gdf['categoria'] = cortes
        else:
            gdf['categoria'] = 'Bajo'
            
        col_name = 'Elevacion'
        
    elif tipo == 'tch':
        # NUEVO: Categorización estricta por rangos fijos para el sensor de rendimiento vryldrcane
        # bins=[-1, 40, 60, 80, 9999] abarca perfectamente: <40, 40 a 60, 60 a 80 y >80
        gdf['categoria'] = pd.cut(
            gdf['vryldrcane'], 
            bins=[-1, 40, 60, 80, 9999], 
            labels=['< 40 Tn/Ha', '40 a 60 Tn/Ha', '60 a 80 Tn/Ha', '> 80 Tn/Ha'], 
            include_lowest=True
        )
        col_name = 'TCH (Sensor)'
        
        
    # --- 1. Generar Plano Visual (Filtrado por distancia_real < 3) ---
    col_filtro = 'distancia_real' if 'distancia_real' in gdf.columns else 'distance'
    gdf_plano = gdf[gdf[col_filtro] < 3].copy()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    minx, miny, maxx, maxy = gdf_plano.total_bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    rango = max(maxx - minx, (maxy - miny) * 2.5) * 1.2
    
    for cat, data in gdf_plano.groupby('categoria'):
        data.plot(ax=ax, color=dict_colores.get(cat, '#7f8c8d'), linewidth=1.5, label=str(cat))
        
    ax.set_xlim(cx - rango / 2, cx + rango / 2)
    ax.set_ylim(cy - (rango / 2.5) / 2, cy + (rango / 2.5) / 2)
    ax.axis('off')
    plt.title(f"PLANO DE {tipo.upper()} DE COSECHA", fontsize=12, fontweight='bold', pad=10)
    plt.legend(loc='lower right', frameon=True)
    plt.savefig(path_img, dpi=100, bbox_inches='tight')
    plt.close()
    
    # --- 2. Generar Cuadro Estadístico (Sin Filtrar - Totalidad de Datos) ---
    gdf['area_ha'] = (gdf['distance'] * gdf['swathwidth']) / 10000
    
    resumen = gdf.groupby('categoria').agg(
        Metros=('distance', 'sum'),
        Hectareas=('area_ha', 'sum')
    ).reset_index()
    
    total_m = resumen['Metros'].sum() if resumen['Metros'].sum() > 0 else 1
    resumen['Distribución (%)'] = (resumen['Metros'] / total_m) * 100
    resumen.columns = [col_name, 'Longitud Total', 'Superficie', 'Distribución (%)']
    
    # Función de pintado exacto de celdas según categoría
    def aplicar_color_categoria(row):
        categoria_fila = row[col_name]
        color_hex = dict_colores.get(categoria_fila, '#ffffff')
        estilos = [''] * len(row)
        idx_distribucion = row.index.get_loc('Distribución (%)')
        
        # El texto va en negro si el fondo es amarillo o verde claro para una óptima lectura
        text_color = 'black' if color_hex in ['#f1c40f', '#2ecc71'] else 'white'
        estilos[idx_distribucion] = f'background-color: {color_hex}; color: {text_color}; font-weight: bold;'
        return estilos

    style = resumen.style.apply(aplicar_color_categoria, axis=1).hide(axis='index')
    style = style.format({
        'Longitud Total': '{:,.1f} m',
        'Superficie': '{:,.2f} ha',
        'Distribución (%)': '{:,.1f} %'
    })
    
    dfi.export(style, path_tabla, table_conversion='chrome')

# --- 3. FUNCIÓN DE RENDERIZADO JINJA ---
def generar_reporte_completo(fila_datos):
    try:
        doc = DocxTemplate(plantilla_path)
        prop = str(fila_datos['Propiedad'].values[0])
        lote = str(fila_datos['Lote'].values[0])
        
        gdf_recorrido = obtener_recorrido_postgres(prop, lote)
        
        # Definición de nombres temporales para planos e imágenes
        p_vel_img, p_vel_tb = "tmp_v.png", "tmp_vt.png"
        p_el_img, p_el_tb = "tmp_e.png", "tmp_et.png"
        p_tch_img, p_tch_tb = "tmp_t.png", "tmp_tt.png"  # NUEVO: TCH
        
        tiene_geo = not gdf_recorrido.empty
        if tiene_geo:
            procesar_y_graficar(gdf_recorrido, 'velocidad', colores_velocidad, p_vel_img, p_vel_tb)
            procesar_y_graficar(gdf_recorrido, 'elevacion', colores_elevacion, p_el_img, p_el_tb)
            procesar_y_graficar(gdf_recorrido, 'tch', colores_tch, p_tch_img, p_tch_tb) # NUEVO: Procesar TCH
        
        # Estructura del Contexto Jinja {{ r.variable }}
        contexto_r = {
            "fecha": datetime.now().strftime("%d/%m/%Y"),
            "propiedad": prop,
            "lote": lote,
            "area_cat": f"{fila_datos['Area Cat'].values[0]:.2f}",
            "area_cos": f"{fila_datos['Area Cosechada'].values[0]:.2f}",
            "avance": str(fila_datos['% Avance'].values[0]),
            "estado": str(fila_datos['Estado'].values[0]),
            "tch_real": f"{fila_datos['TCH Real'].values[0]:.2f}",
            "tn_real": f"{fila_datos['Tn Real'].values[0]:.2f}",
            "tn_limpio": f"{fila_datos['Tn Real Limpio'].values[0]:.2f}",
            "me": f"{fila_datos['ME'].values[0]:.2f} %",
            "pcf": f"{fila_datos['PCF'].values[0]:.2f}",
            "fibra": f"{fila_datos['FIBRA'].values[0]:.2f} %",
            "pureza": f"{fila_datos['PUREZA'].values[0]:.2f} %",
            
            # Inyección de las imágenes en el Word
            "img_plano_vel": InlineImage(doc, p_vel_img, width=Inches(6.0)) if tiene_geo else "[Sin datos]",
            "img_tabla_vel": InlineImage(doc, p_vel_tb, width=Inches(4.5)) if tiene_geo else "",
            "img_plano_el": InlineImage(doc, p_el_img, width=Inches(6.0)) if tiene_geo else "[Sin datos]",
            "img_tabla_el": InlineImage(doc, p_el_tb, width=Inches(4.5)) if tiene_geo else "",
            "img_plano_tch": InlineImage(doc, p_tch_img, width=Inches(6.0)) if tiene_geo else "[Sin datos]", # NUEVO
            "img_tabla_tch": InlineImage(doc, p_tch_tb, width=Inches(4.5)) if tiene_geo else ""              # NUEVO
        }
        
        doc.render({"r": contexto_r})
        
        # Guardado en Unidad G:
        nombre_salida = f"Reporte_Cosecha_Completo_{prop.replace(' ','_')}_{lote.replace(' ','_')}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_salida)
        doc.save(ruta_salida)
        
        # Limpieza de archivos de imagen locales temporales
        for f in [p_vel_img, p_vel_tb, p_el_img, p_el_tb, p_tch_img, p_tch_tb]:
            if os.path.exists(f): os.remove(f)
            
        return f"✅ ¡Reporte completo (Velocidad, Elevación y TCH) generado con éxito!\n📂 Ruta: {ruta_salida}"
        
    except Exception as e:
        return f"❌ Error en la consolidación del reporte: {str(e)}"

# --- 4. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df['Propiedad'].unique()), description='Propiedad:')
combo_lote = widgets.Dropdown(description='Lote:')
btn_generar = widgets.Button(description='Generar Reporte Completo', button_style='success', icon='map')
output_data = widgets.Output()

def actualizar_lotes(*args):
    combo_lote.options = sorted(df[df['Propiedad'] == combo_propiedad.value]['Lote'].unique())

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and combo_lote.value:
            print(f"Lote listo para mapear. Se extraerán mapas de Velocidad, Elevación y TCH (vryldrcane).")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        resultado = df[(df['Propiedad'] == combo_propiedad.value) & (df['Lote'] == combo_lote.value)]
        print("📥 Consultando base de datos PostGIS y renderizando las 3 capas temáticas con Jinja...")
        mensaje = generar_reporte_completo(resultado)
        print(mensaje)

combo_propiedad.observe(actualizar_lotes, 'value')
combo_propiedad.observe(controlar_interfaz, 'value')
combo_lote.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

actualizar_lotes()
display(widgets.HBox([combo_propiedad, combo_lote]), output_data)

Output()